In [ ]:
%pwd

In [ ]:
%cd /oak/stanford/groups/engreitz/Users/tri/Split_pool_Perturb-Seq/analysis/05112023_K562_NC23-24_Mn2/mRNA/mRNA/DGE_filtered

In [ ]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [ ]:
sc.settings.figdir = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230820_eTN2_iPSC_mixed/panmg/BS3/DGE_filtered/"
mat_path = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230820_eTN2_iPSC_mixed/panmg/BS3/DGE_filtered/"
obj_save_path = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230820_eTN2_iPSC_mixed/panmg/BS3/DGE_filtered/"
adata = sc.read_mtx(mat_path + "count_matrix.mtx")



In [ ]:
# reading in gene and cell data
gene_data = pd.read_csv(mat_path + "all_genes.csv")
cell_meta = pd.read_csv(mat_path + "cell_metadata.csv")

# find genes with nan values and filter
gene_data = gene_data[gene_data.gene_name.notnull()]
notNa = gene_data.index
notNa = notNa.to_list()

# remove genes with nan values and assign gene names
adata = adata[:,notNa]
adata.var = gene_data
adata.var.set_index('gene_name', inplace=True)
adata.var.index.name = None
adata.var_names_make_unique()

# add cell meta data to anndata object
adata.obs = cell_meta
adata.obs.set_index('bc_wells', inplace=True)
adata.obs.index.name = None
adata.obs_names_make_unique()

#sc.pp.filter_cells(adata, min_counts=2000)
sc.pp.filter_cells(adata, min_genes=300)
sc.pp.filter_genes(adata, min_cells=5)
adata.shape

In [ ]:
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(6,6), format='png')
#sc.pp.filter_cells(adata, min_counts=50)
#sc.pp.filter_cells(adata, min_genes=20)

In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Scanpy will prepend the string in the save argument with "violin"
# and save it to our figure directory defined in the first step.
sc.pl.violin(adata, ['n_genes_by_counts'], save='_n_genes', jitter=0.4, size = 2)
sc.pl.violin(adata, ['total_counts'], save='_total_counts', jitter=0.4, size = 2)
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4, size = 2)

In [ ]:
# Check cells counts after filtering before assigning
adata[(adata.obs.pct_counts_mt < 15) & 
(adata.obs.n_genes_by_counts < 10000) & 
(adata.obs.total_counts < 100000),:].shape

In [ ]:
# Do the filtering
adata = adata[(adata.obs.pct_counts_mt < 10) & 
(adata.obs.n_genes_by_counts < 10000) & 
(adata.obs.total_counts < 100000),:]

print(adata.shape)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_gene_vs_transcript_counts')
print('median transcript count per cell: ' + str(adata.obs['tscp_count'].median(0)))
print('median gene count per cell: ' + str(adata.obs['gene_count'].median(0)))

In [ ]:
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4)

In [ ]:
adata.shape

In [ ]:
adata.var

In [ ]:
adata_guides = adata[:,[i for (i,x) in enumerate(adata.var.index) if "sg_" in x]]

In [ ]:
adata_guides

In [ ]:
max_cts_pol3 = adata_guides.X.max(axis=1).toarray().astype(int)

In [ ]:
max_cts_pol3

In [ ]:
penguins = sns.load_dataset("penguins")
sns.histplot(data=penguins, x="flipper_length_mm")

In [ ]:
#sns.histplot(data=max_cts, fill=False)

In [ ]:
plt.clf()
plot = sns.histplot(max_cts, log_scale=True, element="step")
#plot = sns.histplot(max_cts, binwidth=3)
plt.rc('font', size=8)
plt.legend("", frameon=False)
plt.xlabel("UMI counts (most abundant guide)")
#plt.savefig(mat_path + "hist.top.all_cells.pdf")
plt.grid(visible=False)
#plt.rc('xtick', labelsize=30)
plt.savefig(mat_path + "hist.top.all_cells.pdf")
plt.show()
plt.clf()

In [ ]:
adata_guide_b08 = adata[:,[i for (i,x) in enumerate(adata.var.index) if "sg_b08" in x]]

In [ ]:
b08=adata_guide_b08.X.toarray().astype(int)

In [ ]:
plot = sns.histplot(b08,binwidth=3)
#plot = sns.histplot(max_cts, binwidth=3)
plt.legend("", frameon=False)
plt.xlabel("UMI counts sg_b08")
plt.savefig(mat_path + "hist.b08.pdf")
plt.clf()

In [ ]:
adata_guide_CDH5 = adata[:,[i for (i,x) in enumerate(adata.var.index) if "sg_CDH5" in x]]

CDH5=adata_guide_CDH5.X.toarray().astype(int)

plot = sns.histplot(CDH5,binwidth=3)
#plot = sns.histplot(max_cts, binwidth=3)
plt.legend("", frameon=False)
plt.xlabel("UMI counts sg_CDH5")
plt.savefig(mat_path + "hist.CDH5.pdf")
plt.clf()

In [ ]:
sns.set_theme(style="white", color_codes=True)
x, y = [], []
for i in range(adata_guides.X.shape[0]):
    row = adata_guides.X[i,:].data
    row = sorted(row)[::-1]
    if len(row)==0:
        continue
    prop = row[0]/np.sum(row)
    y.append(prop)
    x.append(np.sum(row))

sns.scatterplot(x,y)
plt.ylabel("Prop. of reads from most abundant guide")
plt.xlabel("total guide UMIs")
plt.savefig(mat_path + "scatter.pdf")
plt.clf()

In [ ]:
import math
x, y = [], []
for i in range(adata_guides.X.shape[0]):
    row = adata_guides.X[i,:].data
    row = sorted(row)[::-1]
    if len(row)==0:
        continue
    ratio = math.log2(row[0]/row[1])
    y.append(ratio)
    x.append(np.sum(row))

sns.scatterplot(x,y)
plt.ylabel("log2 UMI ratio of top guide over second guide")
plt.xlabel("total guide UMIs")
plt.savefig(mat_path + "top2second_scatter.pdf")
plt.clf()

In [ ]:
sc.settings.figdir = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230621_Tri_guide_iPSCs/iPSC/iPSC_guide/DGE_filtered/"
mat_path = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230621_Tri_guide_iPSCs/iPSC/iPSC_guide/DGE_filtered/"
obj_save_path = "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/230621_Tri_guide_iPSCs/iPSC/iPSC_guide/DGE_filtered/"
adata = sc.read_mtx(mat_path + "DGE.mtx")

In [ ]:
# reading in gene and cell data
gene_data = pd.read_csv(mat_path + "all_genes.csv")
cell_meta = pd.read_csv(mat_path + "cell_metadata.csv")

# find genes with nan values and filter
gene_data = gene_data[gene_data.gene_name.notnull()]
notNa = gene_data.index
notNa = notNa.to_list()

# remove genes with nan values and assign gene names
adata = adata[:,notNa]
adata.var = gene_data
adata.var.set_index('gene_name', inplace=True)
adata.var.index.name = None
adata.var_names_make_unique()

# add cell meta data to anndata object
adata.obs = cell_meta
adata.obs.set_index('bc_wells', inplace=True)
adata.obs.index.name = None
adata.obs_names_make_unique()

#sc.pp.filter_cells(adata, min_counts=2000)
sc.pp.filter_cells(adata, min_genes=300)
sc.pp.filter_genes(adata, min_cells=5)
adata.shape

In [ ]:
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(6,6), format='png')
#sc.pp.filter_cells(adata, min_counts=50)
#sc.pp.filter_cells(adata, min_genes=20)

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Scanpy will prepend the string in the save argument with "violin"
# and save it to our figure directory defined in the first step.
sc.pl.violin(adata, ['n_genes_by_counts'], save='_n_genes', jitter=0.4, size = 2)
sc.pl.violin(adata, ['total_counts'], save='_total_counts', jitter=0.4, size = 2)
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4, size = 2)

In [ ]:
# Check cells counts after filtering before assigning
adata[(adata.obs.pct_counts_mt < 15) & 
(adata.obs.n_genes_by_counts < 10000) & 
(adata.obs.total_counts < 100000),:].shape

In [ ]:
# Do the filtering
adata = adata[(adata.obs.pct_counts_mt < 10) & 
(adata.obs.n_genes_by_counts < 10000) & 
(adata.obs.total_counts < 100000),:]

print(adata.shape)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_gene_vs_transcript_counts')
print('median transcript count per cell: ' + str(adata.obs['tscp_count'].median(0)))
print('median gene count per cell: ' + str(adata.obs['gene_count'].median(0)))

In [ ]:
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4)

In [ ]:
adata.shape

In [ ]:
adata.var

In [ ]:
adata_guides = adata[:,[i for (i,x) in enumerate(adata.var.index) if "b08" in x]]

In [ ]:
adata_guides

In [ ]:
max_cts_pol2 = adata_guides.X.max(axis=1).toarray().astype(int)

In [ ]:
max_cts_pol2

In [ ]:
pol2 = max_cts_pol2[(max_cts_pol2 < 600) & (max_cts_pol2 > 5)]
pol3 = max_cts_pol3[(max_cts_pol3 < 600) & (max_cts_pol3 > 5)]
df2 = pd.DataFrame({'Pol II': pol2})
df3 = pd.DataFrame({'Pol III': pol3})
df2

In [ ]:
df3

In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="darkgrid")
sns.set_context("paper", rc={"font.size":10,"axes.titlesize":10,"axes.labelsize":10}) 
vgf = sns.boxplot(data=[df2, df3])
vgf.set_yticks(range(0,650, 50))
vgf.set(xlabel ="PolII vs PolIII capture", ylabel = "guide UMI per cell")
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)


plt.show()

In [ ]:
np.median(df2)

In [ ]:
np.median(df3)